# Stock News Analyzer - Full Pipeline Notebook

The complete analysis behind the Streamlit app, runnable cell-by-cell in VS Code.

**Course mapping (NMIMS AI 2025-26 Odd Sem):**
- **CO4 / Exp.9** - Supervised learning: decision tree predicting post-news price direction
- **CO4 / Exp.10** - Unsupervised learning: K-means clustering of news text (TF-IDF + SVD)
- **CO3** - Knowledge representation: IF-THEN production rules extracted from the trained tree

**How to run:** open this notebook in VS Code, select the kernel **Python (Stock News Analyzer)**
(it is `.venv\Scripts\python.exe` inside the project folder), then *Run All*.

> All figures are historical observations from the bundled dataset, not predictions and not investment advice.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = pathlib.Path(r"C:\Users\nanda\Downloads\Stock-News-Analyzer-Ollama\stock-news-analyzer")
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 60)

# Plotly figures render inline in the notebook (plotly.js is loaded from CDN).
try:
    from IPython.display import display
    def show_fig(fig):
        if fig is not None:
            display(fig)
except Exception:
    def show_fig(fig):
        print("[plotly figure]", fig.data[0].type if fig and fig.data else "empty")

print("Project root:", ROOT)

## 1. Load and clean the dataset (15,000 news events)

In [ ]:
from src.data_loader import load_and_clean
from src.config import COLUMN_MAP

df, info, path = load_and_clean()
print(f"Loaded {info.rows:,} rows from {path.name}")
print(f"Companies: {len(info.tickers)}  Sectors: {len(info.sectors)}  Topics: {len(info.topics)}")
print(f"Years: {info.date_min} - {info.date_max}  |  Duplicates removed: {info.duplicates_removed}")
df[["news", "year", "company", "sector", "topic", "sentiment", "price_change_percent", "price_direction", "impact"]].head()

## 2. Exploratory statistics

In [ ]:
from src.market import market_impact_stats, news_profile_text

print("Sentiment mix:\n", df[COLUMN_MAP["sentiment"]].value_counts(), sep="\n")
print("\nMost newsworthy topics:\n", df[COLUMN_MAP["event_type"]].value_counts().head(8), sep="\n")
print("\nPrice change (%):\n", df[COLUMN_MAP["price_change_percent"]].describe().round(3), sep="\n")
print("\nMarket impact stats:", market_impact_stats(df))

## 3. The aggregate profile the AI is fed
The app computes these numbers from the CSV itself and injects them into the prompt as a
`<computed_statistics>` block, so the model analyses data instead of restating article text.

In [ ]:
print(news_profile_text(df) or "n/a")
print("\n--- scoped to Infosys ---\n")
print(news_profile_text(df, ["Infosys"]) or "n/a")

## 4. Charts (same Plotly charts as the Dashboard view)

In [ ]:
from src import charts

show_fig(charts.news_count_by_ticker(df))
show_fig(charts.sentiment_over_time(df))
show_fig(charts.price_impact(df))

## 5. Semantic retrieval (sentence-transformers + FAISS)
First run loads the embedding model (cached afterwards) and reuses the pre-built index in `index/news.faiss`.

In [ ]:
from src.retriever import load_embedding_model, Retriever

model = load_embedding_model()
retr = Retriever(df, info, model=model)

results = retr.search("How did Infosys react to earnings news?", k=6)
for r in results:
    print(f"score={r['__score']:.2f}  {r.get('year')}  {r['company']:10s}  {r['sentiment']:9s}  "
          f"{r['price_change_percent']:+6.2f}%  {str(r['news'])[:70]}")

## 6. Grounded answer with the local Ollama model

In [ ]:
from src.config import ai_ready
from src.guardrail import has_sufficient_evidence
from src.analyzer import answer_question

question = "How did Infosys react to earnings news?"
if not has_sufficient_evidence(results):
    print("Guardrail: evidence below similarity threshold - the LLM would not be called.")
elif not ai_ready():
    print("Ollama is not reachable. Start it (`ollama serve`) and install the model: ollama pull llama3.2:3b")
else:
    print(answer_question(question, results, stats_text=news_profile_text(df, ["Infosys"])))

## 7. CO4 / Exp.9 - Supervised learning: decision tree
Predicts post-news price direction (Up / Down / No Change) from sentiment, topic, company,
year and log-volume. Trained on 75%, evaluated on the held-out 25%.

In [ ]:
from src import models

bundle = models.train_direction_model(df)
m = bundle["metrics"]
print(f"Rows used: {bundle['rows_used']:,}  (train {bundle['train_rows']:,} / test {bundle['test_rows']:,})")
print(f"Accuracy: {m['accuracy']:.3f}   Balanced accuracy: {m['balanced_accuracy']:.3f}   "
      f"Majority-class baseline: {m['baseline_majority']:.3f}")
print(f"Dropped rare classes: {bundle['dropped_classes'] or 'none'}")
pd.DataFrame(m["report"]).T[["precision", "recall", "f1-score"]].round(3)

In [ ]:
show_fig(charts.feature_importance(bundle["field_importance"]))
show_fig(charts.confusion_heatmap(bundle["confusion"], bundle["labels"]))

In [ ]:
# What-if prediction from the trained pipeline
pred = models.predict_direction(bundle, {"sentiment": "Positive", "topic": "Earnings", "company": "Infosys"})
print("Predicted direction:", pred["predicted"])
print("Probabilities:", {k: round(v, 3) for k, v in pred["probs"].items()})

## 8. CO3 - Knowledge representation: IF-THEN production rules
The expert-system style rules are extracted directly from the fitted tree (support = number of
training rows reaching the leaf, confidence = share of the predicted class there).

In [ ]:
for r in bundle["rules"][:8]:
    print(f"IF {r['conditions']:<70s} THEN {r['predicted']:<8s} "
          f"(confidence {r['confidence']*100:.0f}%, support {r['support']})")

## 9. CO4 / Exp.10 - Unsupervised learning: K-means on news text
TF-IDF vectors, reduced to 2-D with TruncatedSVD, then clustered. Silhouette is used to pick k.

In [ ]:
quality = models.cluster_quality(df, ks=range(2, 6))
best_k = int(quality.loc[quality["silhouette"].idxmax(), "k"])
show_fig(charts.silhouette_curve(quality))
print("Silhouette by k:", dict(zip(quality["k"], quality["silhouette"].round(3))))
print("Best k =", best_k)

In [ ]:
clus = models.train_clusters(df, n_clusters=best_k)
print(f"k={clus['k']}  silhouette={clus['silhouette']:.3f}  inertia={clus['inertia']:.1f}")
show_fig(charts.cluster_scatter(clus["points"]))
clus["stats"]

## 10. Wrap-up
- The Streamlit app (`streamlit run app.py`) uses exactly these modules: `src.data_loader`,
  `src.retriever`, `src.analyzer`, `src.market`, `src.models`.
- Every number above comes from the bundled CSV; a price move that follows a news item is a
  historical co-occurrence, **not** a prediction and **not** investment advice.